%md
# gold_on_time_delivery

**Fuente:** `workspace.silver.silver_pedidos`
**Destino:** `workspace.gold.gold_on_time_delivery`
**Grano:** `OrderRegion` + `ShippingMode` + `FechaInicio` + `FechaFin` (historizado por corte)

## Qué hace
Equivalente a 2 stored procedures del diseño original, combinados en un solo notebook:
1. `usp_tskInsStage_PerfomanceLogisticoDataCo` (sección Indicador 1) — calcula el % de
   entregas a tiempo, agrupado por región+modo de envío, sobre TODO el histórico acumulado
   en `silver_pedidos`, excluyendo `CANCELED` / `SHIPPING CANCELED`.
2. `usp_tskInsDataMart_OnTimeDelivery` — etiqueta el resultado con el corte
   (`FechaInicio`/`FechaFin`) vigente y lo historiza: `DELETE` de ese corte si ya existía,
   `INSERT` del resultado nuevo.

## Nota de diseño
El `DELETE`+`INSERT` no es atómico entre sí en Delta — si el Job falla justo entre
ambos pasos, ese corte queda sin fila hasta la próxima corrida. Aceptable para
frecuencia semanal.

In [0]:
# ============================================================
# CREACIÓN DEL CATÁLOGO GOLD
# ============================================================

spark.sql("""
CREATE CATALOG IF NOT EXISTS gld_dataco
""")

# ============================================================
# CREACIÓN DEL SCHEMA GOLD
# ============================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS gld_dataco.operaciones
""")

print("✓ Catálogo gld_dataco creado/verificado")
print("✓ Schema gld_dataco.operaciones creado/verificado")

In [0]:
# ============================================================
# PARÁMETROS DEL NOTEBOOK
# ============================================================
#
# Propósito:
#   Definir los parámetros necesarios para identificar los
#   catálogos y esquemas de las capas Silver y Gold.
#
# Arquitectura:
#   Silver → slv_dataco.operaciones
#   Gold   → gld_dataco.operaciones
#
# Tabla Silver origen:
#   slv_dataco.operaciones.hd_pedidos
#
# Tabla Gold destino:
#   gld_dataco.operaciones.kpi_on_time_delivery
#
# Control:
#   El corte de extracción es recibido desde la Task
#   00_prepare_pipeline_control mediante taskValues.
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "operaciones", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "operaciones", "Esquema Gold")

dbutils.widgets.text(
    "control_task_key",
    "00_prepare_pipeline_control",
    "Task Key de control (nombre REAL de la Task en el Job)"
)

# ============================================================
# OBTENER PARÁMETROS
# ============================================================

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

control_task_key = dbutils.widgets.get("control_task_key")

# ============================================================
# DEFINICIÓN DE TABLAS
# ============================================================

SILVER_PEDIDOS_FQN = (
    f"{silver_catalog}.{silver_schema}.hd_pedidos"
)

GOLD_TABLE_FQN = (
    f"{gold_catalog}.{gold_schema}.kpi_on_time_delivery"
)

# ============================================================
# CREAR ESQUEMA GOLD SI NO EXISTE
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}"
)

print(f"✓ Silver origen : {SILVER_PEDIDOS_FQN}")
print(f"✓ Gold destino  : {GOLD_TABLE_FQN}")

In [0]:
# ============================================================
# LECTURA DE SILVER - PEDIDOS
# ============================================================
#
# Propósito:
#   Leer la tabla Silver de pedidos, que contiene el histórico
#   acumulado de órdenes proveniente de Bronze.
#
# Fuente:
#   slv_dataco.operaciones.hd_pedidos
#
# El cálculo del KPI se realizará sobre TODO el histórico
# disponible en Silver.
# ============================================================

df_pedidos = spark.table(SILVER_PEDIDOS_FQN)

print(f"Total filas en hd_pedidos: {df_pedidos.count()}")

# ============================================================
# VALIDACIÓN DE COLUMNAS
# ============================================================

print("Columnas disponibles en hd_pedidos:")
print(df_pedidos.columns)

display(df_pedidos.limit(10))

In [0]:
# ============================================================
# FILTRO DE ÓRDENES VÁLIDAS
# ============================================================
#
# Propósito:
#   Excluir las órdenes canceladas del cálculo del indicador.
#
# Reglas originales:
#   - OrderStatus <> 'CANCELED'
#   - DeliveryStatus <> 'SHIPPING CANCELED'
#
# Las columnas ya fueron normalizadas en Silver a snake_case.
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

df_activo = df_pedidos.filter(
    (F.col("order_status") != "CANCELED") &
    (F.col("delivery_status") != "SHIPPING CANCELED")
)

print(f"Total órdenes válidas: {df_activo.count()}")

# ============================================================
# CÁLCULO DEL INDICADOR ON TIME DELIVERY
# ============================================================
#
# Grano:
#   order_region + shipping_mode
#
# TotalOrdenes:
#   Cantidad total de órdenes válidas.
#
# OrdenesATiempo:
#   Órdenes donde los días reales de envío son menores
#   o iguales a los días programados.
#
# OnTimeDeliveryPct:
#   Porcentaje de órdenes entregadas a tiempo.
# ============================================================

df_indicador = (
    df_activo
    .groupBy(
        "order_region",
        "shipping_mode"
    )
    .agg(
        F.count(F.lit(1)).alias("total_ordenes"),

        F.sum(
            F.when(
                F.col("days_for_shipping_real")
                <= F.col("days_for_shipment_scheduled"),
                1
            ).otherwise(0)
        ).alias("ordenes_a_tiempo")
    )
    .withColumn(
        "on_time_delivery_pct",
        (
            F.col("ordenes_a_tiempo")
            * F.lit(100.0)
            / F.col("total_ordenes")
        ).cast(DecimalType(5, 2))
    )
)

# ============================================================
# VISUALIZACIÓN DEL RESULTADO
# ============================================================

display(df_indicador)

In [0]:
# ============================================================
# OBTENER CORTE DE EXTRACCIÓN
# ============================================================
#
# Propósito:
#   Obtener el rango de fechas correspondiente al corte
#   procesado por el pipeline.
#
# Los valores son publicados por la Task:
#   00_prepare_pipeline_control
#
# Estos valores se utilizan para historizar el KPI en Gold.
# ============================================================

fecha_inicio = dbutils.jobs.taskValues.get(
    taskKey=control_task_key,
    key="fecha_inicio"
)

fecha_fin = dbutils.jobs.taskValues.get(
    taskKey=control_task_key,
    key="fecha_fin"
)

# ============================================================
# VALIDACIÓN DEL CORTE
# ============================================================

if not fecha_inicio or not fecha_fin:
    raise ValueError(
        "No se recibieron fecha_inicio y fecha_fin desde "
        f"la Task '{control_task_key}'."
    )

print(f"✓ FechaInicio : {fecha_inicio}")
print(f"✓ FechaFin    : {fecha_fin}")


# ============================================================
# OBTENER ID DE EJECUCIÓN
# ============================================================
#
# Propósito:
#   Mantener el mismo identificador de ejecución utilizado
#   desde Bronze hasta Gold.
#
# El ID es generado una sola vez por:
#   00_prepare_pipeline_control
#
# No se genera un nuevo UUID en Gold.
# ============================================================

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey=control_task_key,
    key="id_ejecucion"
)

# ============================================================
# VALIDACIÓN DEL ID DE EJECUCIÓN
# ============================================================

if not id_ejecucion:
    raise ValueError(
        "No se recibió id_ejecucion desde "
        f"la Task '{control_task_key}'."
    )

print(f"✓ ID ejecución : {id_ejecucion}")


# ============================================================
# AGREGAR TRAZABILIDAD AL RESULTADO DEL KPI
# ============================================================
#
# El KPI se historiza por:
#   order_region
#   shipping_mode
#   fecha_inicio
#   fecha_fin
#
# Además conserva:
#   fecha_carga
#   id_ejecucion
# ============================================================

df_gold = (
    df_indicador
    .withColumn("fecha_inicio", F.to_date(F.lit(fecha_inicio)))
    .withColumn("fecha_fin", F.to_date(F.lit(fecha_fin)))
    .withColumn("fecha_carga", F.current_timestamp())
    .withColumn("id_ejecucion", F.lit(id_ejecucion))
)

display(df_gold)

In [0]:
# ============================================================
# ESCRITURA DE GOLD - KPI ON TIME DELIVERY
# ============================================================
#
# Propósito:
#   Persistir el resultado del KPI en la capa Gold, historizado
#   por corte (fecha_inicio + fecha_fin).
#
# Estrategia:
#   DELETE de las filas de este corte (si ya existían, por
#   reproceso) + INSERT del resultado nuevo. Esto NO borra
#   cortes anteriores -> necesario para que RESUMEN EJECUTIVO
#   pueda calcular "variación vs semana anterior" (requisito
#   del documento 4_1).
#
#   También resuelve el caso de prueba de idempotencia: correr
#   el mismo corte 2 veces el mismo día debe dejar la tabla
#   idéntica, no duplicada.
#
# Tabla destino:
#   gld_dataco.operaciones.kpi_on_time_delivery
# ============================================================

if spark.catalog.tableExists(GOLD_TABLE_FQN):
    spark.sql(f"""
        DELETE FROM {GOLD_TABLE_FQN}
        WHERE fecha_inicio = DATE('{fecha_inicio}')
          AND fecha_fin    = DATE('{fecha_fin}')
    """)
    print(f"✓ Filas previas del corte {fecha_inicio} → {fecha_fin} eliminadas (si existían)")

(
    df_gold.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(GOLD_TABLE_FQN)
)

print(
    f"✓ {df_gold.count()} filas escritas en "
    f"{GOLD_TABLE_FQN} para el corte {fecha_inicio} → {fecha_fin}"
)